# Aggregates in R — Full Solution

Complete working code, alternate implementations, more-practice solutions, simulation, charts and audience-adapted messaging for the ShoeFly.com aggregates + A/B project.

### Project Flowchart
![Aggregates Flowchart](aggregates_r_flowchart.png)

---
## 0. Setup

In [ ]:
library(readr)
library(dplyr)
library(ggplot2)

orders      <- read_csv("data_aggregates_r/orders.csv")
page_visits <- read_csv("data_aggregates_r/page_visits.csv")
ad_clicks   <- read_csv("data_aggregates_r/ad_clicks.csv")

cat("Data loaded.\n")

### Inspect

In [ ]:
head(orders, 10)
head(page_visits)
head(ad_clicks)
# Optional diagnostics
cat("\nOrders has NA in price? ", any(is.na(orders$price)), "\n")
summary(orders$price)

---
## 1. Overall Column Statistics

In [ ]:
average_price <- orders %>%
  summarize(mean_price = mean(price, na.rm = TRUE))
average_price
# Expected ~ 287.33

most_expensive <- orders %>%
  summarize(max_price = max(price, na.rm = TRUE))
most_expensive
# Expected 470.04

num_colors <- orders %>%
  summarize(n_distinct_shoe_color = n_distinct(shoe_color))
num_colors
# Expected 6

### Alternate code (same result)
```r
# Base R
mean(orders$price, na.rm = TRUE)
max(orders$price, na.rm = TRUE)
length(unique(orders$shoe_color))

# Multiple stats at once
orders %>% summarize(
  mean_p = mean(price, na.rm = TRUE),
  max_p  = max(price, na.rm = TRUE),
  n_col  = n_distinct(shoe_color),
  n_obs  = n()
)
```

---
## 2. One-variable group_by + summarize

In [ ]:
pricey_shoes <- orders %>%
  group_by(shoe_type) %>%
  summarize(max_price = max(price, na.rm = TRUE))
pricey_shoes

shoes_sold <- orders %>%
  group_by(shoe_type) %>%
  summarize(count = n())
shoes_sold

### Alternate
```r
# tidyverse count() shortcut
orders %>% count(shoe_type, name = "count")

# Base R
tapply(orders$price, orders$shoe_type, max, na.rm = TRUE)
table(orders$shoe_type)
```

---
## 3. Multi-column grouping

In [ ]:
shoe_counts <- orders %>%
  group_by(shoe_type, shoe_color) %>%
  summarize(count = n(), .groups = "drop")
shoe_counts

shoe_prices <- orders %>%
  group_by(shoe_type, shoe_material) %>%
  summarize(mean_price = mean(price, na.rm = TRUE), .groups = "drop")
shoe_prices

---
## 4. group_by + filter (per-group metric)

In [ ]:
most_pop_orders <- orders %>%
  group_by(shoe_type) %>%
  filter(n() > 16)
most_pop_orders

# Verify which types survived
most_pop_orders %>% count(shoe_type)

---
## 5. group_by + mutate (keep all rows)

In [ ]:
diff_from_mean <- orders %>%
  group_by(shoe_type) %>%
  mutate(diff_from_shoe_type_mean = price - mean(price, na.rm = TRUE))

# Show a few rows
diff_from_mean %>%
  select(shoe_type, price, diff_from_shoe_type_mean) %>%
  head(12)

---
## 6. Page-visit traffic

In [ ]:
average_price <- orders %>%
  summarize(mean_price = mean(price, na.rm = TRUE))
average_price

click_source <- page_visits %>%
  group_by(utm_source) %>%
  summarize(count = n())
click_source

click_source_by_month <- page_visits %>%
  group_by(utm_source, month) %>%
  summarize(count = n(), .groups = "drop")
click_source_by_month

---
## 7. A/B Ad-Click Analysis

In [ ]:
# 7.1 Views by platform
views_by_utm <- ad_clicks %>%
  group_by(utm_source) %>%
  summarize(count = n())
views_by_utm

# 7.2 Clicks / non-clicks
clicks_by_utm <- ad_clicks %>%
  group_by(utm_source, ad_clicked) %>%
  summarize(count = n(), .groups = "drop")
clicks_by_utm

# 7.3 Percentage
percentage_by_utm <- clicks_by_utm %>%
  group_by(utm_source) %>%
  mutate(percentage = count / sum(count))
percentage_by_utm

clicked_pct <- percentage_by_utm %>%
  filter(ad_clicked == TRUE) %>%
  arrange(desc(percentage))
clicked_pct
# facebook & google lead (~35.7 % / 35.1 %); email & twitter lag (~31 %)

In [ ]:
# 7.4 Experiment balance & A vs B
experiment_split <- ad_clicks %>%
  group_by(experimental_group) %>%
  summarize(count = n())
experiment_split
# roughly 828 A / 826 B — well balanced

clicks_by_experiment <- ad_clicks %>%
  group_by(experimental_group, ad_clicked) %>%
  summarize(count = n(), .groups = "drop")
clicks_by_experiment

ab_pct <- clicks_by_experiment %>%
  group_by(experimental_group) %>%
  mutate(percentage = count / sum(count)) %>%
  filter(ad_clicked == TRUE)
ab_pct
# Ad A CTR ≈ 287/828 ≈ 34.7 %; Ad B CTR ≈ 278/826 ≈ 33.7 % → slight edge for A

In [ ]:
# 7.5 Day-of-week analysis
a_clicks <- ad_clicks %>% filter(experimental_group == "A")
b_clicks <- ad_clicks %>% filter(experimental_group == "B")

a_clicks_by_day <- a_clicks %>%
  group_by(day, ad_clicked) %>%
  summarize(count = n(), .groups = "drop")

b_clicks_by_day <- b_clicks %>%
  group_by(day, ad_clicked) %>%
  summarize(count = n(), .groups = "drop")

a_percentage_by_day <- a_clicks_by_day %>%
  group_by(day) %>%
  mutate(percentage = count / sum(count))

b_percentage_by_day <- b_clicks_by_day %>%
  group_by(day) %>%
  mutate(percentage = count / sum(count))

a_clicked_day <- a_percentage_by_day %>% filter(ad_clicked == TRUE)
b_clicked_day <- b_percentage_by_day %>% filter(ad_clicked == TRUE)

cat("=== Ad A daily CTR ===\n")
print(a_clicked_day %>% select(day, percentage) %>% arrange(day))
cat("\n=== Ad B daily CTR ===\n")
print(b_clicked_day %>% select(day, percentage) %>% arrange(day))

**Recommendation:** Overall Ad A has a modestly higher CTR. Day-of-week variation exists; check whether the gap is stable or driven by one or two days before declaring a permanent winner. Facebook and Google are the strongest traffic sources for clicks.

---
## 8. More Practice — Solutions

In [ ]:
# 1. Median & IQR of price by shoe_material
orders %>%
  group_by(shoe_material) %>%
  summarize(
    median_price = median(price, na.rm = TRUE),
    iqr_price    = IQR(price, na.rm = TRUE),
    .groups = "drop"
  )

# 2. shoe_type with highest average price
orders %>%
  group_by(shoe_type) %>%
  summarize(avg = mean(price, na.rm = TRUE), .groups = "drop") %>%
  slice_max(avg, n = 1)

# 3. Proportion paid-social (facebook + twitter) vs rest
page_visits %>%
  mutate(channel = if_else(utm_source %in% c("facebook", "twitter"), "paid_social", "other")) %>%
  count(channel) %>%
  mutate(prop = n / sum(n))

# 4. Flag high-CTR platforms (CTR > 0.33) then count rows
high_ctr_platforms <- percentage_by_utm %>%
  filter(ad_clicked == TRUE, percentage > 0.33) %>%
  pull(utm_source)

ad_clicks %>%
  mutate(high_ctr = utm_source %in% high_ctr_platforms) %>%
  count(high_ctr)

# 5. Base-R version of click percentage
tab <- table(ad_clicks$utm_source, ad_clicks$ad_clicked)
prop.table(tab, margin = 1)   # row-wise proportions

---
## 9. Simulation Section

In [ ]:
simulate_aggregates <- function(orders_df = orders,
                                ad_df = ad_clicks,
                                min_orders = 16,
                                sample_frac = 1.0,
                                price_noise_sd = 0,
                                seed = 42) {
  set.seed(seed)
  
  # optional price noise
  o <- orders_df
  if (price_noise_sd > 0) {
    o$price <- o$price + rnorm(nrow(o), 0, price_noise_sd)
  }
  
  # popular shoe types under current threshold
  popular <- o %>%
    group_by(shoe_type) %>%
    filter(n() > min_orders) %>%
    ungroup() %>%
    count(shoe_type, name = "n_orders")
  
  # sub-sample ads
  ad_sub <- ad_df %>% sample_frac(sample_frac)
  
  ctr <- ad_sub %>%
    group_by(utm_source, ad_clicked) %>%
    summarize(count = n(), .groups = "drop") %>%
    group_by(utm_source) %>%
    mutate(percentage = count / sum(count)) %>%
    filter(ad_clicked == TRUE) %>%
    select(utm_source, ctr = percentage)
  
  ab <- ad_sub %>%
    group_by(experimental_group, ad_clicked) %>%
    summarize(count = n(), .groups = "drop") %>%
    group_by(experimental_group) %>%
    mutate(pct = count / sum(count)) %>%
    filter(ad_clicked == TRUE) %>%
    select(experimental_group, ctr = pct)
  
  list(
    n_ad_rows = nrow(ad_sub),
    popular_shoes = popular,
    ctr_by_source = ctr,
    ab_ctr = ab
  )
}

# Baseline
cat("=== Baseline (min_orders=16, full sample) ===\n")
print(simulate_aggregates())

# Stricter popularity threshold
cat("\n=== Stricter threshold min_orders=40 ===\n")
print(simulate_aggregates(min_orders = 40))

# Smaller experiment (50 % sample)
cat("\n=== 50 % sample of ad_clicks ===\n")
print(simulate_aggregates(sample_frac = 0.5, seed = 123))

# Price noise sensitivity
cat("\n=== Price + noise sd=30 ===\n")
print(simulate_aggregates(price_noise_sd = 30)$popular_shoes)

---
## 10. Visual Summary

In [ ]:
# Max / mean price by shoe_type
price_summary <- orders %>%
  group_by(shoe_type) %>%
  summarize(
    mean_price = mean(price, na.rm = TRUE),
    max_price  = max(price, na.rm = TRUE),
    .groups = "drop"
  )

p1 <- ggplot(price_summary, aes(x = reorder(shoe_type, mean_price), y = mean_price)) +
  geom_col(fill = "#3498db") +
  geom_point(aes(y = max_price), color = "#e74c3c", size = 3) +
  coord_flip() +
  labs(title = "Mean (bars) and Max (red dots) Price by Shoe Type",
       x = NULL, y = "Price ($)") +
  theme_minimal()
print(p1)

# Click rate by utm_source
p2 <- ggplot(clicked_pct, aes(x = reorder(utm_source, percentage), y = percentage)) +
  geom_col(fill = "#27ae60") +
  scale_y_continuous(labels = scales::percent) +
  coord_flip() +
  labs(title = "Click-Through Rate by Traffic Source",
       x = NULL, y = "CTR") +
  theme_minimal()
print(p2)

# A vs B overall
p3 <- ggplot(ab_pct, aes(x = experimental_group, y = percentage, fill = experimental_group)) +
  geom_col(width = 0.5) +
  scale_y_continuous(labels = scales::percent) +
  labs(title = "Overall CTR: Ad A vs Ad B", x = "Ad Version", y = "CTR") +
  theme_minimal() + theme(legend.position = "none")
print(p3)

---
## 11. Audience-Adapted Takeaways

### 1. Executive / Decision-Maker (low data-literacy, high stakes)
> **Recommendation:** Keep Ad A as the default creative. It delivered a ~1 percentage-point higher click-through rate than Ad B on a well-balanced test of ~1 650 impressions. Facebook and Google remain the highest-performing traffic sources; consider shifting more budget there. One missing price record in the order file did not affect the conclusion.

### 2. Data Analyst / Peer (high data-literacy)
> Pipeline used: `group_by(utm_source, ad_clicked) %>% summarize(n()) %>% group_by(utm_source) %>% mutate(pct = n/sum(n))`. Overall A CTR = 287/828 ≈ 34.7 %, B = 278/826 ≈ 33.7 %. Day-of-week CTRs fluctuate; a formal binomial or logistic test (or Bayesian A/B) is advisable before permanent rollout. Missing price handled with `na.rm = TRUE`. Simulation shows the A>B edge is stable down to 50 % sample size.

### 3. Nonspecialist Stakeholder (marketing intern / store associate)
> An “aggregate” is simply a single number that summarises many numbers—like the average price of all shoes sold, or the percentage of people who clicked an ad. We ran a fair race between two ad designs (A and B). A won by a small but consistent margin. People who arrived from Facebook or Google were more likely to click than people who arrived from email or Twitter. That is why we suggest using Ad A and focusing our ads on those two platforms.

---
## Key Takeaways (for the 1-page report)

- `summarize()` collapses rows; `group_by()` + `summarize()` collapses **per group**.
- `group_by()` + `filter(summary_fn)` keeps/drops whole groups of rows based on a group metric.
- `group_by()` + `mutate(summary_fn)` adds a group-level calculation while preserving every original row.
- Always decide `na.rm` behaviour when missing values exist.
- A/B click-rate analysis follows the same pattern: group → count → percentage = count/sum(count) → filter to the event of interest.
- Audience adaptation (exec vs analyst vs nonspecialist) changes vocabulary, level of detail and recommended next action—not the underlying arithmetic.